# 예제 02. 문장 길이 맞추기
빅데이터프로그래밍 · 12주차

## 목표
- 길이가 다른 문장을 같은 길이로 맞춘다
- `<PAD>` 가 왜 0번인지 이해한다
- Dataset과 DataLoader로 batch를 만든다

batch로 묶으려면 모든 문장의 길이가 같아야 합니다. 11주차의 시점 축과 같은 문제입니다.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import re
from collections import Counter

PAD, UNK = 0, 1

def tokenize(s):
    s = s.lower()
    s = re.sub(r"[^가-힣a-z0-9\s]", " ", s)
    return s.split()

sentences = [
    "재미있다",
    "이 영화는 정말 재미있었다",
    "배우들의 연기가 훌륭하고 이야기도 탄탄해서 오래 기억에 남을 작품이었다",
    "지루했다",
    "시간 낭비였고 다시 보고 싶지 않다",
]
labels = [1, 1, 1, 0, 0]

tokens = [tokenize(s) for s in sentences]
for t in tokens:
    print(len(t), t)


길이가 1부터 11까지 제각각입니다. 이대로는 Tensor로 묶을 수 없습니다.


## 1. 사전 만들기


In [ ]:
counter = Counter(w for t in tokens for w in t)
vocab = {"<PAD>": PAD, "<UNK>": UNK}
for w, _ in counter.most_common():
    vocab[w] = len(vocab)

def encode(t):
    return [vocab.get(w, UNK) for w in t]

ids = [encode(t) for t in tokens]
for i in ids:
    print(len(i), i)


## 2. 길이 맞추기 — 짧으면 0으로 채우고, 길면 자릅니다


In [ ]:
MAX_LEN = 8

def pad(seq, max_len=MAX_LEN):
    seq = seq[:max_len]                       # 길면 자르기
    return seq + [PAD] * (max_len - len(seq))  # 짧으면 0으로 채우기

for s, i in zip(sentences, ids):
    print(f"{pad(i)}   ← {s[:20]}")


모두 길이 8이 되었습니다. 0은 "여기는 비었다"는 뜻입니다.


In [ ]:
x = torch.tensor([pad(i) for i in ids])
print("shape:", tuple(x.shape), "→ (문장 수, 최대 길이)")


## 3. 앞을 채우기 vs 뒤를 채우기
RNN은 마지막 시점으로 예측하므로, 뒤를 0으로 채우면 마지막이 빈칸이 됩니다.


In [ ]:
def pad_front(seq, max_len=MAX_LEN):
    seq = seq[-max_len:]
    return [PAD] * (max_len - len(seq)) + seq

short = ids[0]
print("뒤를 채움:", pad(short), "← 마지막이 0")
print("앞을 채움:", pad_front(short), "← 마지막이 실제 단어")


짧은 문장이 많다면 앞을 채우는 편이 낫습니다. 실습에서는 뒤를 채우고 `<PAD>` 를 무시하도록 처리합니다.


## 4. MAX_LEN 을 얼마로 할까
너무 짧으면 정보를 버리고, 너무 길면 빈칸만 늘어납니다.


In [ ]:
import numpy as np
import pandas as pd

lens = [len(t) for t in tokens]
print("길이:", lens)
print(f"평균 {np.mean(lens):.1f} · 중앙값 {np.median(lens):.1f} · 최대 {max(lens)}")

rows = []
for m in [4, 8, 16]:
    cut = sum(1 for l in lens if l > m)
    padded = sum(max(0, m - l) for l in lens)
    rows.append({"MAX_LEN": m, "잘린 문장 수": cut, "채운 빈칸 총합": padded})
pd.DataFrame(rows)


보통 **길이 분포의 90~95 분위**를 씁니다.


## 5. Dataset과 DataLoader


In [ ]:
class TextDataset(Dataset):
    def __init__(self, ids, labels, max_len=MAX_LEN):
        self.x = torch.tensor([pad(i, max_len) for i in ids])
        self.y = torch.tensor(labels)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.x[i], self.y[i]


ds = TextDataset(ids, labels)
loader = DataLoader(ds, batch_size=3, shuffle=True)

xb, yb = next(iter(loader))
print("입력:", tuple(xb.shape), "→ (batch, 길이)")
print("정답:", tuple(yb.shape))
print("\n", xb)


## 6. PAD를 무시하도록 알려주기
`padding_idx=0` 을 주면 임베딩이 0번을 학습하지 않습니다.


In [ ]:
emb_no = nn.Embedding(len(vocab), 8)
emb_pad = nn.Embedding(len(vocab), 8, padding_idx=PAD)

print("padding_idx 없음, 0번 벡터:", emb_no.weight[0].data.round(decimals=3))
print("padding_idx 지정, 0번 벡터:", emb_pad.weight[0].data.round(decimals=3), "← 전부 0")


## 직접 해보기
1. `MAX_LEN=4` 로 하면 어떤 문장이 잘리나요?
2. 문장 길이의 90 분위를 계산해 `MAX_LEN` 으로 써 보세요.
3. 앞을 채우는 방식으로 Dataset을 고쳐 보세요.


In [ ]:
# 여기에 작성하세요
